In [1]:
%pip install openai pandas openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 6.4 MB/s  0:00:0036m-:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [openai]2m1/2 [openai]
Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install --upgrade pydantic openai

  Attempting uninstall: pydantic
    Found existing installation: pydantic 2.12.4
    Uninstalling pydantic-2.12.4:
      Successfully uninstalled pydantic-2.12.4
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd
import json
import time
import os
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from openai import OpenAI

# ================= 1. 配置区 =================
API_KEY = "sk-66abb0de87e74cd2a52d9a3e487dce5c"
client = OpenAI(
    api_key=API_KEY,
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1"
)
MODEL_NAME = "qwen-plus"
INPUT_FILE = "/Users/syx/Desktop/5508/副本news_cleaned_2025_for_llm.xlsx"
OUTPUT_FILE = "/Users/syx/Desktop/5508/news_mo_extracted.xlsx"
MAX_WORKERS = 5

# ================= 2. Prompt =================
SYSTEM_PROMPT = """你是一名犯罪脚本分析编码员（Crime Script Analyst）。你的任务是将每篇新闻编码为标准化的诈骗MO（Modus Operandi）行为脚本，供后续无监督机器学习聚类使用。

【纳入标准】
仅当新闻描述了具体的诈骗事件或诈骗手法，且包含可识别的行为细节时，is_scam = true。
以下情况 is_scam = false：纯政策法规报道、纯判决量刑报道（无行为描述）、一般性防骗提醒（无具体手法）、与诈骗无关的新闻。

【编码原则】
1. 提取可观测的行为动作，不是案件摘要、不是传统诈骗类别名称。
2. 每个MO维度选择1个主标签（primary）；仅当确实存在第二关键行为时填写次标签（secondary），否则填该维度的 X0 标签。
3. 标签必须完整输出"前缀+汉字"，如 "TRU1_公权身份伪装"，绝不能只输出汉字。
4. 标签字段中禁止出现任何具体品牌名、机构名、人名、金额、地名。
5. 多语种新闻统一使用简体中文标签。行为机制相同则必须归入同一标签。

【七个核心MO维度标签库】

1. PREP — 前期准备与选靶（骗子在接触受害者之前做了什么准备）
PREP1_人设身份伪造：创建虚假身份、包装社交媒体资料、伪造职业背景
PREP2_平台网站搭建：搭建虚假投资平台、购物网站、任务平台、钓鱼页面
PREP3_数据名单获取：购买或窃取个人信息数据库、通过泄露数据定向筛选目标
PREP4_无明显前期准备：机会主义型，无明显前期基础设施建设
PREP0_未提及：新闻未提供前期准备信息

2. CONTACT — 初始接触方式（骗子如何与受害者建立第一次联系）
CON1_盲发广撒触达：群发短信、群发邮件、机器人电话、批量添加好友
CON2_社交平台搭讪：在社交媒体或交友平台主动搭讪、私信、点赞互动
CON3_需求场景切入：在受害者主动寻找工作/贷款/服务时出现并回应需求
CON4_冒名定向联络：假借官方/银行/快递/熟人名义，直接联系特定个人
CON5_线下物理接触：面对面搭话、线下推广、实体店面诱导
CON6_受害者主动上门：受害者通过搜索引擎、广告、应用商店自行找到骗子
CON0_未提及：新闻未说明如何建立接触

3. TRUST — 信任建构机制（骗子用什么手段让受害者相信自己）
TRU1_公权身份伪装：冒充警察、检察官、法官、税务、海关等政府公权力
TRU2_机构品牌冒用：冒充银行、电商平台、快递公司、运营商等企业客服
TRU3_熟人关系利用：冒充或利用亲友、同事、同学、上级等已有社会关系
TRU4_专业人设包装：打造投资导师、理财专家、成功商人等专业人设
TRU5_群体氛围伪造：建立假群聊、安排托儿展示收益、伪造用户评价
TRU6_小额返利验证：先给予真实小额回报让受害者亲身验证"可信"和"可盈利"
TRU7_伪造凭证文件：伪造银行流水截图、合同文件、身份证件、授权书等
TRU0_未提及：新闻未说明信任建构方式

4. MANIPULATION — 操控与推进触发（骗子用什么手段推动受害者采取行动）
MAN1_恐吓威胁施压：声称涉及犯罪、面临逮捕、账户冻结、法律后果等
MAN2_高收益利诱：承诺高额回报、丰厚佣金、大额奖励
MAN3_情感绑架操控：利用恋爱感情、亲情、同情心、愧疚感来推动行动
MAN4_制造紧急时限：强调"立刻""今天必须""名额有限""马上到期"
MAN5_隔离保密要求：明确要求不得告诉家人朋友、不得报警、单独行动
MAN6_沉没成本追加：利用已投入的资金或时间，诱导受害者继续追加以求挽回
MAN0_未提及：新闻未说明操控手段

5. OPERATION — 操作诱导行为（骗子要求受害者执行的具体技术操作）
OPR1_下载安装应用：要求下载特定APP、安装指定软件
OPR2_共享屏幕远程控制：要求开启屏幕共享、安装远程桌面工具
OPR3_点击链接填写信息：诱导点击钓鱼链接、在虚假页面填入账号密码等
OPR4_注册账户加入群组：要求在特定平台注册账户或加入指定群聊
OPR5_执行刷单任务：要求完成刷单、点赞、下单等"任务"操作
OPR6_上传证件人脸识别：要求上传身份证照片、进行人脸识别认证
OPR0_未提及：新闻未说明具体操作诱导

6. EXTRACTION — 价值转移通道（资金或价值如何从受害者转移到骗子）
EXT1_银行转账：通过银行账户直接转账汇款
EXT2_第三方数字支付：通过支付宝、微信支付、PayPal等第三方平台支付
EXT3_加密货币转移：通过比特币、USDT等加密货币转移
EXT4_礼品卡充值卡：要求购买礼品卡、充值卡并提供卡号密码
EXT5_线下现金交割：面对面交付现金
EXT6_账户权限接管：骗子获取受害者银行/支付账户的直接控制权
EXT0_未提及：新闻未说明资金转移方式

7. AFTERMATH — 案后行为特征（资金转移后骗子的行为）
AFT1_立即失联消失：立刻拉黑、删除联系方式、关闭账号
AFT2_设障拖延拒付：以各种理由拒绝提现、拖延出金
AFT3_编造新由追骗：编造手续费、税费、保证金等新理由继续要求付款
AFT4_转化身份复害：更换身份后再次接触同一受害者进行二次诈骗
AFT5_转为勒索威胁：利用已获取的隐私信息或影像资料进行勒索
AFT6_发展为工具人：将受害者发展为帮助洗钱、收款、拉人等工具角色
AFT0_未提及：新闻未说明案后行为

【辅助字段——不参与聚类，用于后续分析画像】

surface_scenario（表面场景，仅选一个）：
投资理财 / 刷单返利 / 冒充公权 / 冒充客服 / 冒充熟人 / 恋爱交友 / 招聘兼职 / 贷款征信 / 虚假交易 / 色情勒索 / 技术支持 / 网络赌博 / 其他

psychological_vulnerability（受害者心理脆弱点，仅选一个）：
P1_权威服从 / P2_损失恐惧 / P3_贪婪侥幸 / P4_情感孤独 / P5_生存焦虑 / P6_隐私羞耻 / P0_未提及

compliance_driver（受害者行动驱动力，仅选一个）：
D1_避免惩罚 / D2_挽回损失 / D3_获取收益 / D4_维持关系 / D5_保护名誉 / D6_解决急需 / D0_未提及

detail_quality（行为细节丰富度）：
高：可明确编码5个及以上MO维度
中：可明确编码3-4个MO维度
低：仅知道是诈骗，但行为链条模糊（≤2个维度可编码）

reasoning_process：50-120字，简述你的编码判断逻辑。
key_evidence_quotes：提取1-2句新闻原文中体现核心手段的短语。
script_track_summary：15-40字，用自然语言概括整条行为链，不含任何具体名称和金额。

【scene_sequence 构建规则】
1. 用完整的前缀标签（如 "PREP1_人设身份伪造"）构建一条3-8步时序链条。
2. 按照行为发生的先后顺序排列。
3. 允许回退或循环（如追骗场景可重复出现 MAN6 和 EXT）。
4. 示例：
["PREP1_人设身份伪造", "CON2_社交平台搭讪", "TRU4_专业人设包装", "TRU6_小额返利验证", "MAN2_高收益利诱", "OPR1_下载安装应用", "EXT3_加密货币转移", "MAN6_沉没成本追加", "AFT2_设障拖延拒付"]

【输出格式——严格遵守】
1. 严格输出JSON，禁止Markdown代码块，禁止输出JSON以外的任何文本。
2. 若 is_scam=false，仅输出：{"is_scam": false}
3. 若 is_scam=true，输出以下完整结构：

{
  "is_scam": true,
  "reasoning_process": "",
  "prep_primary": "",
  "prep_secondary": "",
  "contact_primary": "",
  "contact_secondary": "",
  "trust_primary": "",
  "trust_secondary": "",
  "manipulation_primary": "",
  "manipulation_secondary": "",
  "operation_primary": "",
  "operation_secondary": "",
  "extraction_primary": "",
  "extraction_secondary": "",
  "aftermath_primary": "",
  "aftermath_secondary": "",
  "scene_sequence": [],
  "script_track_summary": "",
  "surface_scenario": "",
  "detail_quality": "",
  "psychological_vulnerability": "",
  "compliance_driver": "",
  "key_evidence_quotes": ""
}
"""

# ================= 3. 字段列表 =================
MO_COLUMNS = [
    "is_scam", "reasoning_process",
    "prep_primary", "prep_secondary",
    "contact_primary", "contact_secondary",
    "trust_primary", "trust_secondary",
    "manipulation_primary", "manipulation_secondary",
    "operation_primary", "operation_secondary",
    "extraction_primary", "extraction_secondary",
    "aftermath_primary", "aftermath_secondary",
    "scene_sequence", "script_track_summary",
    "surface_scenario", "detail_quality",
    "psychological_vulnerability", "compliance_driver",
    "key_evidence_quotes"
]

# ================= 4. API调用（含重试） =================
def extract_mo(news_text, max_retries=3):
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": f"请提取以下新闻的诈骗行为脚本特征：\n\n{news_text[:4000]}"}
                ],
                stream=False,
                response_format={"type": "json_object"},
                temperature=0.1,
            )
            raw = response.choices[0].message.content.strip()
            result = json.loads(raw)

            # 布尔值修正
            if isinstance(result.get("is_scam"), str):
                result["is_scam"] = result["is_scam"].lower() == "true"

            # scene_sequence: list → 字符串
            seq = result.get("scene_sequence", [])
            if isinstance(seq, list):
                result["scene_sequence"] = " → ".join(str(v) for v in seq)

            return result

        except json.JSONDecodeError:
            print(f"  ⚠️ JSON解析失败 (尝试 {attempt+1}/{max_retries})")
            time.sleep(2)
        except Exception as e:
            print(f"  ❌ API错误: {e} (尝试 {attempt+1}/{max_retries})")
            time.sleep(3)

    return {"is_scam": False}

# ================= 5. 主程序 =================
if __name__ == "__main__":

    # --- 断点续传 ---
    if os.path.exists(OUTPUT_FILE):
        print(f"📂 发现已有输出文件，断点续传: {OUTPUT_FILE}")
        try:
            df = pd.read_excel(OUTPUT_FILE)
        except Exception as e:
            print(f"❌ 读取失败: {e}，请手动删除输出文件后重试")
            exit()
    else:
        print(f"📂 全新启动，读取: {INPUT_FILE}")
        try:
            df = pd.read_excel(INPUT_FILE)
        except FileNotFoundError:
            print(f"❌ 找不到文件: {INPUT_FILE}")
            exit()

    # --- 初始化列 ---
    for col in MO_COLUMNS:
        if col not in df.columns:
            df[col] = ""

    # --- 找出待处理行 ---
    indices_to_process = []
    for idx, row in df.iterrows():
        val = row.get("is_scam")
        if pd.isna(val) or str(val).strip() == "":
            indices_to_process.append(idx)

    total = len(df)
    remaining = len(indices_to_process)
    print(f"📊 总计: {total} | 待处理: {remaining} | 已完成: {total - remaining}")

    if remaining == 0:
        print("🎉 所有数据已处理完毕！")
        exit()

    print(f"🚀 启动并发提取 (线程数: {MAX_WORKERS})...\n")

    data_lock = threading.Lock()
    done_count = 0

    def process_row(index):
        text = str(df.at[index, "llm_input"])
        if len(text) < 30 or text.lower() == "nan":
            return index, {"is_scam": False}
        return index, extract_mo(text)

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(process_row, i): i for i in indices_to_process}

        for future in as_completed(futures):
            idx = futures[future]
            try:
                _, result = future.result()
                is_scam = result.get("is_scam", False)

                with data_lock:
                    done_count += 1
                    df.at[idx, "is_scam"] = is_scam

                    if is_scam:
                        for col in MO_COLUMNS[1:]:
                            val = result.get(col, "未提及")
                            if isinstance(val, (dict, list)):
                                df.at[idx, col] = json.dumps(val, ensure_ascii=False)
                            else:
                                df.at[idx, col] = str(val) if val else "未提及"
                    else:
                        for col in MO_COLUMNS[1:]:
                            df.at[idx, col] = "不适用"

                    print(f"✅ {done_count}/{remaining} (行{idx+2}, scam={is_scam})")

                    if done_count % 20 == 0:
                        df.to_excel(OUTPUT_FILE, index=False)
                        print(f"  💾 已保存 ({done_count}条)")

            except Exception as exc:
                print(f"❌ 行{idx+2} 异常: {exc}")

    df.to_excel(OUTPUT_FILE, index=False)
    scam_count = df["is_scam"].apply(lambda x: str(x).strip().lower() in ["true", "1"]).sum()
    print(f"\n🎉 全部完成！保存至 {OUTPUT_FILE}")
    print(f"   总计 {total} 条 | 识别为诈骗: {scam_count} 条")

In [3]:
import pandas as pd
import json
import time
from openai import OpenAI

# ================= 1. 配置区 =================
# ⚠️ 注意：为了安全，请务必保护好你的 API_KEY
API_KEY = "sk-66abb0de87e74cd2a52d9a3e487dce5c"

client = OpenAI(
    api_key=API_KEY, 
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1"
)

MODEL_NAME = "qwen-plus"

INPUT_FILE = "/Users/syx/Desktop/5508/副本news_cleaned_2025_for_llm.xlsx"   
OUTPUT_FILE = "news_structured_result_new.xlsx" 
# =========================================================

def extract_fraud_structure(news_text):
    system_prompt = """
你是一名“诈骗犯罪脚本分析编码员（crime script analyst）”。你的任务不是只给新闻贴上传统诈骗类别，也不是总结新闻，而是将每篇新闻编码为一个“通用诈骗 MO 脚本”，用于后续的 bottom-up 聚类与多维映射分析。

【方法定位】
本任务借鉴 crime script analysis（CSA）的思路，但不照搬任何特定诈骗类型（如 romance scam）的题材内容。
你的目标是：
1. 将诈骗过程压缩编码为 6 个核心场景；
2. 识别每个场景中最关键的主动作；
3. 识别诈骗中的心理脆弱点、激活方式与受害者响应；
4. 保留诈骗过程的非线性、反馈循环与复害特征；
5. 识别该诈骗脚本中哪些环节更容易被自动化、哪些环节更依赖人工接管；
6. 不预设最终诈骗大类，最终模式应由后续聚类从脚本组合中归纳。

【重要区分】
本任务参考的是一般化的 MO / script 结构，而不是 romance scam 的题材内容。
因此：
- 不要默认存在恋爱关系、情感承诺、送礼、婚姻规划、性暗示等 romance-scam 专属内容；
- 只有新闻明确提及时，才可作为具体 open_action_code 或 summary 中的可选变体；
- 任何 romance-specific 行为都不能作为一般诈骗的默认结构。

【多语种处理总则】
本任务面向多语种新闻数据。输入可能为中文、英文、法文、德文、土耳其文、俄文、西班牙文、葡萄牙文、印尼文等任意语言，也可能出现混合语言、转写文本、翻译文本或机器翻译痕迹。

处理原则如下：
1. 必须优先理解原始新闻语言中的标题与正文，再参考译文辅助判断；
2. 若同时提供原文与译文，原文优先，译文仅作辅助解释，不得无条件覆盖原文；
3. 所有输出标签必须统一归一为简体中文；
4. 不得因为语言不同、机构名称不同、平台名称不同而生成不同标签；
5. 地名、职业名、文化叙事、宗教表述、国家机关名称，只要底层欺骗机制一致，就必须归入同一标准标签；
6. 若因语言歧义、翻译不完整、文本残缺导致无法稳定判断，优先填“不明确”；
7. 不允许在输出中保留原文外语短语作为主标签；
8. 若仅 key_evidence_quotes 或 open_action_code 中必须保留原文含义，也必须先翻译后概括为简体中文（或保留外语并加括号备注中文）。

【输入字段优先级规则】
若输入中包含 language、title、content、translated_content、combined_text 等字段，请按以下顺序理解：
1. title + content = 主判断依据
2. translated_content = 辅助判断依据
3. combined_text = 补充上下文，不得替代主判断
4. language 字段用于帮助判断语境和翻译风险，但不得直接决定标签

当原文与译文存在差异时：
- 优先依据原文判断；
- 若原文过短或难以理解，可参考译文；
- 若两者冲突明显且无法消解，相关字段填“不明确”。

【6 个核心场景】
1. setup_targeting_action：前期准备与选靶
2. contact_isolation_action：接触与转入可控沟通空间
3. trust_conditioning_action：建信与关系/权威塑造
4. manipulation_trigger_action：操控推进与关键触发
5. transfer_extraction_action：资金/资产/价值转移
6. continuation_aftermath_action：持续榨取、复害或收尾

【研究对象纳入规则】
只有当新闻满足以下条件时，is_scam=true：
A. 存在明确的诱导过程；
B. 诱导目标与财产、支付、资产、账号、信息价值、转账或经济利益相关；
C. 行为通过互动过程推进，而不是单纯暴力夺取或单纯系统攻击。

【排除规则】
以下情况通常 is_scam=false：
1. 普通盗窃、抢劫、谋杀、绑架；
2. 单纯腐败、洗钱、走私、逃税；
3. 单纯文件伪造、商业欺诈、合同纠纷，但没有清晰诱导链；
4. 单纯黑客攻击、数据泄露，但没有诈骗式互动过程；
5. 只有抓捕、通报、政策、警示，没有清晰 MO。

【字段编码总原则】
1. 你是在做“脚本化编码”，不是做长摘要。
2. 每个主字段只输出一个主标签。
3. 不允许在单字段中写多个并列标签，不得写“A/B”“A并B”“A+B”。
4. 案件整体允许复合结构，这种组合关系应通过 scene_sequence 表达。
5. 核心字段必须使用标准化简体中文标签，不得输出英文、品牌名、平台名、人名、国家名、金额。
6. 若新闻未提及，填“未提及”；若提及但不足以判断，填“不明确”。
7. 任何 romance scam 专属动作都不能默认出现，只有在文本中明确出现时才可写入 open_action_code。
8. 若新闻提到多个案件，只编码信息最多、行为链最完整的主案件。
9. 若同一阶段中出现多个动作，优先选择“最推动骗局进展的主动作”。
10. 若文本只是背景介绍或统计综述，而非具体个案过程，应降低 detail_quality。

【跨语言归一化规则】
你需要关注的是“底层诈骗机制”，而不是不同语言中的表面说法。请执行以下归一化：
1. 不同国家和语言中的官方/机构名，只要核心作用是制造权威、施压、核验等，统一优先归入：冒充官方 / 法律恐吓 / 制造异常 等对应标签。
2. 不同语言中的支付名目（如 customs fee、tax fee），只要本质是以“付款才能继续”推进骗局，统一优先归入：制造危机 / 要求支付手续费 / 转账付款 等。
3. 不同语言中的高收益叙事、投资平台等，统一归入：利益诱导 / 投资入金 / 虚拟币转移 / 转入虚假平台 等。
4. 不同语言中的特定人设（医生、海外军人等），核心是建立可信人设，统一优先判断为：专业包装 / 冒充熟人 / 长期养熟 / 展示凭证 等。
5. 任何文化、国家、职业差异，都不能直接生成新分类；底层 MO 不同时，才允许用 open_action_code 补充。

【心理维度编码原则】
心理字段描述的是“被骗过程中被利用的心理脆弱点”，不是对受害者人格的永久判断。请避免责备受害者的表达。

【自动化与工业化分析原则】
本任务需要额外判断：
1. 该诈骗脚本是否适合被批量化、自动化或半自动化执行；
2. 哪个阶段最适合被规模化放大；
3. 哪些阶段更需要人工接管与实时调整；
4. 当受害者出现迟疑、抗拒或拒绝时，骗子是否表现出动态调整与升级策略。
注意：
- 前期低上下文、重复性高、模板化强的阶段，自动化潜力通常更高；
- 长程建信、复杂人设维护、抗拒应对、动态升级通常更依赖人工；
- 若新闻没有足够信息，不要强猜，填“不明确”。

【特别提醒：多语种数据中的常见错误】
1. 不要把不同语言中的同义机构、同义危机、同义费用误判为不同诈骗类型；
2. 不要把具体职业、国家机关、文化叙事当成独立主标签；
3. 不要因为使用了本地化叙事，就忽略其底层 MO 与心理利用机制；
4. 不要因为译文措辞更完整，就直接抛弃原文；
5. 若新闻只是转述或摘要，关键过程不完整，应降低 detail_quality 与 crosslingual_confidence。

【标准标签集合】

一、六个核心场景主动作字段

1. setup_targeting_action
可选：伪造身份 / 准备账号 / 准备话术 / 准备平台 / 广泛撒网 / 精准筛选 / 脆弱性筛选 / 信息摸排 / 重复选靶 / 未提及 / 不明确

2. contact_isolation_action
可选：假借通知 / 主动搭讪 / 批量群发 / 平台应答 / 招聘邀约 / 熟人切入 / 电话接触 / 线下搭话 / 转入私聊 / 转入即时通讯 / 转入外部链接 / 转入虚假平台 / 未提及 / 不明确

3. trust_conditioning_action
可选：冒充官方 / 冒充客服 / 冒充熟人 / 冒充招聘方 / 展示凭证 / 伪造收益 / 专业包装 / 长期养熟 / 群体背书 / 未提及 / 不明确

4. manipulation_trigger_action
可选：法律恐吓 / 制造异常 / 制造危机 / 利益诱导 / 情感操控 / 时间施压 / 要求保密 / 小额返利诱导 / 持续追加要求 / 限制退出 / 未提及 / 不明确

5. transfer_extraction_action
可选：银行转账 / 第三方支付 / 虚拟币转移 / 礼品卡支付 / 现金交付 / 账号接管 / 投资入金 / 代购代付 / 账户接力转移 / 未提及 / 不明确

6. continuation_aftermath_action
可选：拒绝提现 / 追损骗局 / 信息转卖 / 再次收割 / 换身份复联 / 失联消失 / 受害者识破退出 / 敲诈勒索 / 转向其他骗局 / 未提及 / 不明确

二、心理维度

7. targeted_psychological_vulnerability
可选：权威服从 / 损失恐惧 / 机会贪求 / 情感依赖 / 社交信任 / 求职焦虑 / 资金焦虑 / 隐私羞耻 / 稀缺焦虑 / 救急心理 / 未提及 / 不明确

8. psychological_activation_action
可选：制造权威压力 / 制造损失风险 / 展示高额收益 / 提供情感陪伴 / 假借熟人关系 / 强调稀缺机会 / 制造紧急情境 / 放大羞耻后果 / 制造账户异常 / 展示虚假成功 / 未提及 / 不明确

9. compliance_driver
可选：避免惩罚 / 挽回损失 / 获取收益 / 维持关系 / 保护名誉 / 获得工作 / 获得贷款 / 服从权威 / 解决紧急问题 / 未提及 / 不明确

三、受害者响应

10. victim_response_action
可选：提供信息 / 下载操作 / 转账付款 / 持续加码 / 配合保密 / 转移渠道 / 线下交付 / 未提及 / 不明确

四、辅助字段

11. surface_scenario
可选：投资理财 / 刷单返利 / 冒充公权 / 冒充客服 / 冒充熟人 / 恋爱交友 / 招聘兼职 / 贷款征信 / 虚假交易 / 色情引流 / 中奖福利 / 其他诈骗 / 未提及 / 不明确

12. detail_quality
可选：高 / 中 / 低

五、自动化与工业化辅助字段

13. automation_suitability
可选：高 / 中 / 低 / 不明确

14. human_oversight_needed
可选：高 / 中 / 低 / 不明确

15. industrialisation_leverage_point
可选：前期选靶 / 初始触达 / 私聊隔离 / 建信脚本 / 心理画像 / 资金转移 / 追骗复害 / 未提及 / 不明确

16. adaptive_resistance_present
可选：是 / 否 / 不明确

17. feedback_loop_present
可选：是 / 否 / 不明确

18. revictimization_present
可选：是 / 否 / 不明确

19. crosslingual_confidence
可选：高 / 中 / 低

六、质量控制与溯源辅助字段

20. reasoning_process
含义：在输出其他标签前，简要写出你对该案件阶段推进、心理利用、工业化特征及跨语言映射的分析逻辑（50-150字）。强制要求第一步输出。

21. key_evidence_quotes
含义：提取 1-2 句最能体现核心诈骗手段的原文短语（若是外语请保留外语原词并附简要翻译），用于人工复核信度。

【scene_sequence 要求】
1. 按实际推进顺序输出 4–7 个阶段化动作；
2. 体现诈骗脚本推进，不要写传统诈骗类别名；
3. 允许出现回退或循环；
4. 每一步尽量复用标准动作标签；
5. 若信息不足，置为空数组。
示例：["准备账号", "批量群发", "转入私聊", "冒充官方", "制造异常", "银行转账"]

【script_track_summary 要求】
1. 用 20–35 字概括该诈骗脚本；
2. 不要写成 romance scam 专属叙事；
3. 突出“接触—建信—操控—转移价值”的主链；
4. 尽量体现心理弱点利用。

【开放字段】
open_action_code：仅当出现标准标签无法覆盖的重要动作时填写；长度不超过8个字；必须是简体中文；若无必要，填“无”。

【detail_quality 判定】
高：6 个核心场景大部分可判断，心理至少 1 项明确，自动化字段至少 1 项可判断
中：多个关键场景可判断，但存在明显缺失
低：仅能判断发生诈骗，但脚本结构不完整

【输出要求】
严格输出 JSON，严禁输出 Markdown 代码块。
不要输出解释，除了 JSON 之外不要输出任何额外文本。
注意：JSON 键名请严格对照以下结构，尤其是 reasoning_process 必须排在首位以确保思维链生效。

【JSON结构】
{
  "is_scam": true,
  "reasoning_process": "",
  "setup_targeting_action": "",
  "contact_isolation_action": "",
  "trust_conditioning_action": "",
  "manipulation_trigger_action": "",
  "transfer_extraction_action": "",
  "continuation_aftermath_action": "",
  "targeted_psychological_vulnerability": "",
  "psychological_activation_action": "",
  "compliance_driver": "",
  "victim_response_action": "",
  "automation_suitability": "",
  "human_oversight_needed": "",
  "industrialisation_leverage_point": "",
  "adaptive_resistance_present": "",
  "feedback_loop_present": "",
  "revictimization_present": "",
  "crosslingual_confidence": "",
  "scene_sequence": [],
  "script_track_summary": "",
  "surface_scenario": "",
  "detail_quality": "",
  "open_action_code": "",
  "key_evidence_quotes": ""
}
    """
    
    result_text = ""
    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"请逐维度提取以下新闻中的行为特征：\n\n{news_text}"}
            ],
            stream=False,
            response_format={"type": "json_object"},
            temperature=0.1  # 保持低温确保结构稳定性
        )
        result_text = response.choices[0].message.content.strip()
        result = json.loads(result_text)

        # 防御：字符串 "true"/"false" 转布尔值
        if isinstance(result.get("is_scam"), str):
            result["is_scam"] = result["is_scam"].lower() == "true"

        return result
        
    except json.JSONDecodeError as e:
        print(f"⚠️ JSON 解析失败: {e}")
        print(f"   原始返回: {result_text[:300]}")
        return {"is_scam": False}
    except Exception as e:
        print(f"❌ API 调用错误: {e}")
        return {"is_scam": False}

# ================= 2. 主程序 =================
if __name__ == "__main__":
    print(f"📂 正在读取: {INPUT_FILE} ...")
    try:
        df = pd.read_excel(INPUT_FILE)
    except FileNotFoundError:
        print(f"❌ 找不到文件 {INPUT_FILE}")
        exit()

    # 🌟 字段已与最新 Prompt 中的 JSON 结构完全对齐，包含工业化及跨语言维度
    new_columns = [
        "is_scam", 
        "reasoning_process", 
        "setup_targeting_action", 
        "contact_isolation_action", 
        "trust_conditioning_action", 
        "manipulation_trigger_action", 
        "transfer_extraction_action", 
        "continuation_aftermath_action", 
        "targeted_psychological_vulnerability", 
        "psychological_activation_action", 
        "compliance_driver", 
        "victim_response_action", 
        "automation_suitability", 
        "human_oversight_needed", 
        "industrialisation_leverage_point", 
        "adaptive_resistance_present", 
        "feedback_loop_present", 
        "revictimization_present", 
        "crosslingual_confidence", 
        "scene_sequence", 
        "script_track_summary", 
        "surface_scenario", 
        "detail_quality", 
        "open_action_code", 
        "key_evidence_quotes"
    ]
    
    for col in new_columns:
        if col not in df.columns:
            df[col] = ""

    total_rows = len(df)
    print(f"🚀 共 {total_rows} 条，调用 {MODEL_NAME} 提取行为特征...")

    for index, row in df.iterrows():
        news_content = str(row.get('llm_input', ''))

        if len(news_content) < 30 or news_content.lower() == 'nan':
            df.at[index, 'is_scam'] = False
            for col in new_columns[1:]:
                df.at[index, col] = "不适用"
            continue

        print(f"🔄 第 {index + 1}/{total_rows} 条...")
        extracted = extract_fraud_structure(news_content)

        is_scam = extracted.get('is_scam', False)
        df.at[index, 'is_scam'] = is_scam

        if is_scam:
            for col in new_columns[1:]:
                val = extracted.get(col, '未提及')
                
                # 序列化处理：对行为序列列表(scene_sequence)及意外字典做字符串处理
                if col == "scene_sequence" and isinstance(val, list):
                    df.at[index, col] = " -> ".join(str(v) for v in val) 
                elif isinstance(val, dict) or isinstance(val, list):
                    # 兜底：万一大模型其它字段返回了数组/字典，不至于让 DataFrame 报错
                    df.at[index, col] = json.dumps(val, ensure_ascii=False)
                else:
                    df.at[index, col] = str(val)
        else:
            for col in new_columns[1:]:
                df.at[index, col] = "不适用"

        # 控制并发请求速度，保护 API
        time.sleep(0.5)

        if (index + 1) % 20 == 0:
            df.to_excel(OUTPUT_FILE, index=False)
            print(f"   💾 已保存前 {index + 1} 条")

    df.to_excel(OUTPUT_FILE, index=False)
    print(f"\n🎉 运行完毕！基于最新满配版架构的高质量数据已存入: {OUTPUT_FILE}")

📂 正在读取: /Users/syx/Desktop/5508/副本news_cleaned_2025_for_llm.xlsx ...
🚀 共 18978 条，调用 qwen-plus 提取行为特征...
🔄 第 1/18978 条...
🔄 第 2/18978 条...
🔄 第 3/18978 条...
🔄 第 4/18978 条...
🔄 第 5/18978 条...
🔄 第 6/18978 条...
🔄 第 7/18978 条...
🔄 第 8/18978 条...
🔄 第 9/18978 条...
🔄 第 10/18978 条...
🔄 第 11/18978 条...
🔄 第 12/18978 条...
🔄 第 13/18978 条...
🔄 第 14/18978 条...
🔄 第 15/18978 条...
🔄 第 16/18978 条...
🔄 第 17/18978 条...
🔄 第 18/18978 条...
🔄 第 19/18978 条...
🔄 第 20/18978 条...
   💾 已保存前 20 条
🔄 第 21/18978 条...
🔄 第 22/18978 条...
🔄 第 23/18978 条...
🔄 第 24/18978 条...
🔄 第 25/18978 条...
🔄 第 26/18978 条...
🔄 第 27/18978 条...
🔄 第 28/18978 条...
🔄 第 29/18978 条...
🔄 第 30/18978 条...
🔄 第 31/18978 条...
🔄 第 32/18978 条...
🔄 第 33/18978 条...
🔄 第 34/18978 条...
🔄 第 35/18978 条...
🔄 第 36/18978 条...
🔄 第 37/18978 条...
🔄 第 38/18978 条...
🔄 第 39/18978 条...
🔄 第 40/18978 条...
   💾 已保存前 40 条
🔄 第 41/18978 条...
🔄 第 42/18978 条...
🔄 第 43/18978 条...
🔄 第 44/18978 条...
🔄 第 45/18978 条...
🔄 第 46/18978 条...
🔄 第 47/18978 条...
🔄 第 48/18978 条...
🔄 第 49/1897

KeyboardInterrupt: 

In [ ]:
import pandas as pd
import json
import time
import os
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from openai import OpenAI

# ================= 1. 配置区 =================
# ⚠️ 注意：请填入你自己的阿里云 API Key
API_KEY = "sk-66abb0de87e74cd2a52d9a3e487dce5c"

client = OpenAI(
    api_key=API_KEY, 
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1"
)

MODEL_NAME = "qwen-plus"

INPUT_FILE = "/Users/syx/Desktop/5508/副本news_cleaned_2025_for_llm.xlsx"   
OUTPUT_FILE = "/Users/syx/Desktop/5508/news_structured_result_new.xlsx"
# ⚠️ 并发线程数：建议设为 5
MAX_WORKERS = 5
# =========================================================

def extract_fraud_structure(news_text):
    system_prompt = """
你是一名“诈骗犯罪脚本分析编码员（crime script analyst）”。你的任务不是只给新闻贴上传统诈骗类别，也不是总结新闻，而是将每篇新闻编码为一个“通用诈骗 MO 脚本”，用于后续的 bottom-up 聚类与多维映射分析。

【方法定位】
本任务借鉴 crime script analysis（CSA）的思路，但不照搬任何特定诈骗类型（如 romance scam）的题材内容。
你的目标是：
1. 将诈骗过程压缩编码为 6 个核心场景；
2. 识别每个场景中最关键的主动作；
3. 识别诈骗中的心理脆弱点、激活方式与受害者响应；
4. 保留诈骗过程的非线性、反馈循环与复害特征；
5. 识别该诈骗脚本中哪些环节更容易被自动化、哪些环节更依赖人工接管；
6. 不预设最终诈骗大类，最终模式应由后续聚类从脚本组合中归纳。

【重要区分】
- 不要默认存在恋爱关系、情感承诺、送礼、婚姻规划、性暗示等 romance-scam 专属内容；
- 只有新闻明确提及时，才可作为具体 open_action_code 或 summary 中的可选变体；
- 任何 romance-specific 行为都不能作为一般诈骗的默认结构。

【多语种处理总则】
本任务面向多语种新闻数据。
1. 必须优先理解原始新闻语言中的标题与正文，再参考译文辅助判断；
2. 所有输出标签必须统一归一为简体中文；
3. 不得因为语言不同、机构名称不同、平台名称不同而生成不同标签；
4. 地名、职业名、文化叙事、宗教表述、国家机关名称，只要底层欺骗机制一致，就必须归入同一标准标签；
5. 不允许在输出中保留原文外语短语作为主标签（仅 key_evidence_quotes 允许保留外语并备注）。

【6 个核心场景】
1. setup_targeting_action：前期准备与选靶
2. contact_isolation_action：接触与转入可控沟通空间
3. trust_conditioning_action：建信与关系/权威塑造
4. manipulation_trigger_action：操控推进与关键触发
5. transfer_extraction_action：资金/资产/价值转移
6. continuation_aftermath_action：持续榨取、复害或收尾

【研究对象纳入规则】
只有当新闻满足以下条件时，is_scam=true：
A. 存在明确的诱导过程；
B. 诱导目标与财产、支付、资产、账号、信息价值、转账或经济利益相关；
C. 行为通过互动过程推进，而不是单纯暴力夺取或单纯系统攻击。

【排除规则】
以下情况通常 is_scam=false：
1. 普通盗窃、抢劫、谋杀、绑架；
2. 单纯腐败、洗钱、走私、逃税；
3. 单纯文件伪造、商业欺诈、合同纠纷，但没有清晰诱导链；
4. 单纯黑客攻击、数据泄露，但没有诈骗式互动过程；
5. 只有抓捕、通报、政策、警示，没有清晰 MO。

【字段编码总原则】
1. 核心字段原则上只能填写一个“主标签”，不得填写多个标签并列（不得写A/B）。
2. 严格区分动作边界，不得将同一行为放入错误字段。例如：
   - “熟人切入”是接触入口，只能放 contact_isolation_action。
   - “长期养熟”是信任机制，只能放 trust_conditioning_action。
   - “要求资金验证/缴纳保证金”是诱导理由，只能放 manipulation_trigger_action。
   - 真实的物理/数字资金转移（如“银行转账”），才能放入 transfer_extraction_action。
3. 若新闻未提及，填“未提及”；若提及但不足以判断，填“不明确”。

【自动化与工业化分析原则】
判断诈骗脚本是否适合被批量化、自动化或半自动化执行。
- 前期低上下文、重复性高、模板化强的阶段，自动化潜力通常更高；
- 长程建信、复杂人设维护、抗拒应对、动态升级通常更依赖人工。

【标准标签集合】

一、六个核心场景主动作字段
1. setup_targeting_action 可选：伪造身份 / 准备账号 / 准备话术 / 准备平台 / 广泛撒网 / 精准筛选 / 脆弱性筛选 / 信息摸排 / 重复选靶 / 未提及 / 不明确
2. contact_isolation_action 可选：假借通知 / 主动搭讪 / 批量群发 / 平台应答 / 招聘邀约 / 熟人切入 / 电话接触 / 线下搭话 / 转入私聊 / 转入即时通讯 / 转入外部链接 / 转入虚假平台 / 未提及 / 不明确
3. trust_conditioning_action 可选：冒充官方 / 冒充客服 / 冒充熟人 / 冒充招聘方 / 展示凭证 / 伪造收益 / 专业包装 / 长期养熟 / 群体背书 / 未提及 / 不明确
4. manipulation_trigger_action 可选：法律恐吓 / 制造异常 / 制造危机 / 利益诱导 / 情感操控 / 时间施压 / 要求保密 / 小额返利诱导 / 持续追加要求 / 限制退出 / 未提及 / 不明确
5. transfer_extraction_action 可选：银行转账 / 第三方支付 / 虚拟币转移 / 礼品卡支付 / 现金交付 / 账号接管 / 投资入金 / 代购代付 / 账户接力转移 / 未提及 / 不明确
6. continuation_aftermath_action 可选：拒绝提现 / 追损骗局 / 信息转卖 / 再次收割 / 换身份复联 / 失联消失 / 受害者识破退出 / 敲诈勒索 / 转向其他骗局 / 未提及 / 不明确

二、心理维度
7. targeted_psychological_vulnerability 可选：权威服从 / 损失恐惧 / 机会贪求 / 情感依赖 / 社交信任 / 求职焦虑 / 资金焦虑 / 隐私羞耻 / 稀缺焦虑 / 救急心理 / 未提及 / 不明确
8. psychological_activation_action 可选：制造权威压力 / 制造损失风险 / 展示高额收益 / 提供情感陪伴 / 假借熟人关系 / 强调稀缺机会 / 制造紧急情境 / 放大羞耻后果 / 制造账户异常 / 展示虚假成功 / 未提及 / 不明确
9. compliance_driver 可选：避免惩罚 / 挽回损失 / 获取收益 / 维持关系 / 保护名誉 / 获得工作 / 获得贷款 / 服从权威 / 解决紧急问题 / 未提及 / 不明确

三、受害者响应
10. victim_response_action 可选：提供信息 / 下载操作 / 转账付款 / 持续加码 / 配合保密 / 转移渠道 / 线下交付 / 未提及 / 不明确

四、辅助字段
11. surface_scenario 可选：投资理财 / 刷单返利 / 冒充公权 / 冒充客服 / 冒充熟人 / 恋爱交友 / 招聘兼职 / 贷款征信 / 虚假交易 / 色情引流 / 中奖福利 / 其他诈骗 / 未提及 / 不明确
12. detail_quality 可选：高 / 中 / 低

五、自动化与工业化辅助字段
13. automation_suitability 可选：高 / 中 / 低 / 不明确
14. human_oversight_needed 可选：高 / 中 / 低 / 不明确
15. industrialisation_leverage_point 可选：前期选靶 / 初始触达 / 私聊隔离 / 建信脚本 / 心理画像 / 资金转移 / 追骗复害 / 未提及 / 不明确
16. adaptive_resistance_present 可选：是 / 否 / 不明确
17. feedback_loop_present 可选：是 / 否 / 不明确
18. revictimization_present 可选：是 / 否 / 不明确
19. crosslingual_confidence 可选：高 / 中 / 低

六、质量控制与溯源辅助字段
20. reasoning_process 含义：在输出其他标签前，简要写出分析逻辑（50-150字）。强制要求第一步输出。
21. key_evidence_quotes 含义：提取 1-2 句体现核心诈骗手段的原文短语用于复核。

【scene_sequence 要求】
1. 按推进顺序输出 4–7 个动作，允许回退或循环；
2. 示例：["准备账号", "批量群发", "转入私聊", "冒充官方", "制造异常", "银行转账"]

【输出要求】
严格输出 JSON，严禁输出 Markdown 代码块。
除了 JSON 之外不要输出任何额外文本。
⚠️ 【短路规则】：如果在【研究对象纳入规则】判断下该新闻不属于诈骗案件（is_scam=false），请直接仅输出 {"is_scam": false}，绝对不要输出其余任何字段！

【JSON结构】
如果 is_scam 为 true，请严格输出以下完整结构：
{
  "is_scam": true,
  "reasoning_process": "",
  "setup_targeting_action": "",
  "contact_isolation_action": "",
  "trust_conditioning_action": "",
  "manipulation_trigger_action": "",
  "transfer_extraction_action": "",
  "continuation_aftermath_action": "",
  "targeted_psychological_vulnerability": "",
  "psychological_activation_action": "",
  "compliance_driver": "",
  "victim_response_action": "",
  "automation_suitability": "",
  "human_oversight_needed": "",
  "industrialisation_leverage_point": "",
  "adaptive_resistance_present": "",
  "feedback_loop_present": "",
  "revictimization_present": "",
  "crosslingual_confidence": "",
  "scene_sequence": [],
  "script_track_summary": "",
  "surface_scenario": "",
  "detail_quality": "",
  "open_action_code": "",
  "key_evidence_quotes": ""
}

如果 is_scam 为 false，直接输出：
{
  "is_scam": false
}
    """
    
    result_text = ""
    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"请逐维度提取以下新闻中的行为特征：\n\n{news_text}"}
            ],
            stream=False,
            response_format={"type": "json_object"},
            temperature=0.1
        )
        result_text = response.choices[0].message.content.strip()
        result = json.loads(result_text)

        if isinstance(result.get("is_scam"), str):
            result["is_scam"] = result["is_scam"].lower() == "true"

        return result
        
    except json.JSONDecodeError as e:
        print(f"⚠️ JSON 解析失败: {e}\n原文摘要: {news_text[:50]}")
        return {"is_scam": False}
    except Exception as e:
        print(f"❌ API 调用错误: {e}")
        return {"is_scam": False}

# ================= 2. 主程序 (全新启动 + 安全保存 + 多线程) =================
if __name__ == "__main__":
    
    print(f"📂 从源文件 {INPUT_FILE} 全新启动...")
    try:
        df = pd.read_excel(INPUT_FILE)
    except FileNotFoundError:
        print(f"❌ 找不到源文件 {INPUT_FILE}")
        exit()

    # 字段对齐检查
    new_columns = [
        "is_scam", "reasoning_process", "setup_targeting_action", "contact_isolation_action", 
        "trust_conditioning_action", "manipulation_trigger_action", "transfer_extraction_action", 
        "continuation_aftermath_action", "targeted_psychological_vulnerability", 
        "psychological_activation_action", "compliance_driver", "victim_response_action", 
        "automation_suitability", "human_oversight_needed", "industrialisation_leverage_point", 
        "adaptive_resistance_present", "feedback_loop_present", "revictimization_present", 
        "crosslingual_confidence", "scene_sequence", "script_track_summary", "surface_scenario", 
        "detail_quality", "open_action_code", "key_evidence_quotes"
    ]
    
    for col in new_columns:
        if col not in df.columns:
            df[col] = "" # 填充缺失的新列

    # 获取所有行索引以进行全量处理
    indices_to_process = list(df.index)
    total_rows = len(df)
    print(f"📊 总数据量: {total_rows} | 准备全量处理...")

    if total_rows == 0:
        print("源文件为空，程序退出。")
        exit()

    print(f"🚀 启动并发处理 (最大线程数: {MAX_WORKERS})...")

    # 定义单行处理函数
    def process_row(index):
        news_content = str(df.at[index, 'llm_input'])
        
        # 文本过短直接当做非诈骗跳过
        if len(news_content) < 30 or news_content.lower() == 'nan':
            return index, {"is_scam": False} 
            
        extracted = extract_fraud_structure(news_content)
        return index, extracted

    # 创建数据锁，确保写操作是线程安全的
    data_lock = threading.Lock()
    processed_in_this_run = 0

    # 线程池并发执行
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        future_to_index = {executor.submit(process_row, i): i for i in indices_to_process}
        
        for future in as_completed(future_to_index):
            index = future_to_index[future]
            
            try:
                # 解包元组
                _, extracted = future.result() 
                is_scam = extracted.get('is_scam', False)
                
                # 获取锁，安全地写入 DataFrame 并保存文件
                with data_lock:
                    processed_in_this_run += 1
                    df.at[index, 'is_scam'] = is_scam

                    if is_scam:
                        # 诈骗，提取所有字段
                        for col in new_columns[1:]:
                            val = extracted.get(col, '未提及')
                            if col == "scene_sequence" and isinstance(val, list):
                                df.at[index, col] = " -> ".join(str(v) for v in val) 
                            elif isinstance(val, (dict, list)):
                                df.at[index, col] = json.dumps(val, ensure_ascii=False)
                            else:
                                df.at[index, col] = str(val)
                    else:
                        # 非诈骗，填不适用
                        for col in new_columns[1:]:
                            df.at[index, col] = "不适用"
                            
                    print(f"✅ 完成 {processed_in_this_run}/{total_rows} (行号: {index+2}, 是否诈骗: {is_scam})")
                    
                    # ========================================================
                    # ⚠️ 20条直接安全保存机制 (加了线程锁，绝对安全)
                    # ========================================================
                    if processed_in_this_run % 20 == 0:
                        df.to_excel(OUTPUT_FILE, index=False)
                        print(f"   💾 [安全保存] 已成功更新文件: {OUTPUT_FILE}")
                        
            except Exception as exc:
                print(f"❌ 行号 {index+2} 处理产生致命异常: {exc}")

    # 循环结束后，最后做一次全量保存
    df.to_excel(OUTPUT_FILE, index=False)
    print(f"\n🎉 运行彻底完毕！所有数据均已安全存入: {OUTPUT_FILE}")

📂 从源文件 /Users/syx/Desktop/5508/副本news_cleaned_2025_for_llm.xlsx 全新启动...
📊 总数据量: 18978 | 准备全量处理...
🚀 启动并发处理 (最大线程数: 5)...
✅ 完成 1/18978 (行号: 3, 是否诈骗: False)
✅ 完成 2/18978 (行号: 4, 是否诈骗: False)
✅ 完成 3/18978 (行号: 8, 是否诈骗: False)
✅ 完成 4/18978 (行号: 7, 是否诈骗: False)
✅ 完成 5/18978 (行号: 2, 是否诈骗: False)
✅ 完成 6/18978 (行号: 6, 是否诈骗: False)
✅ 完成 7/18978 (行号: 9, 是否诈骗: False)
✅ 完成 8/18978 (行号: 10, 是否诈骗: False)
✅ 完成 9/18978 (行号: 13, 是否诈骗: False)
✅ 完成 10/18978 (行号: 14, 是否诈骗: False)
✅ 完成 11/18978 (行号: 11, 是否诈骗: False)
✅ 完成 12/18978 (行号: 16, 是否诈骗: False)
✅ 完成 13/18978 (行号: 18, 是否诈骗: False)
✅ 完成 14/18978 (行号: 17, 是否诈骗: False)
✅ 完成 15/18978 (行号: 19, 是否诈骗: False)
✅ 完成 16/18978 (行号: 21, 是否诈骗: False)
✅ 完成 17/18978 (行号: 20, 是否诈骗: False)
✅ 完成 18/18978 (行号: 22, 是否诈骗: False)
✅ 完成 19/18978 (行号: 23, 是否诈骗: False)
✅ 完成 20/18978 (行号: 25, 是否诈骗: False)
   💾 [安全保存] 已成功更新文件: /Users/syx/Desktop/5508/news_structured_result_new.xlsx
✅ 完成 21/18978 (行号: 26, 是否诈骗: False)
✅ 完成 22/18978 (行号: 27, 是否诈骗: False)
✅ 完成 23/18978 (行号: 28, 是否诈